# TensorFlow Privacy を使用した差分プライバシーの実装

## 学習目標

* TensorFlow Privacy を使用して、既存のオプティマイザ（例: SGD、Adam）を差分プライバシー対応のオプティマイザにラップする方法を学ぶ
* 差分プライベート機械学習によって導入されるハイパーパラメータを理解する
* TensorFlow Privacy に含まれる分析ツールを使用して、提供されるプライバシー保証を測定する

## 概要

[差分プライバシー（Differential Privacy: DP）](https://en.wikipedia.org/wiki/Differential_privacy) は、アルゴリズムによって提供されるプライバシー保証を測定するためのフレームワークです。差分プライバシーの観点を通じて、プライベートなデータに対して責任を持ってモデルをトレーニングする機械学習アルゴリズムを設計できます。差分プライバシーを用いた学習は測定可能なプライバシー保証を提供し、機械学習において機微なトレーニングデータが公開されるリスクを軽減するのに役立ちます。直感的には、差分プライバシーでトレーニングされたモデルは、データセット内の単一のトレーニングサンプル（または小さなサンプルのセット）によって影響を受けるべきではありません。これにより、機械学習において機微なトレーニングデータが漏洩するリスクを軽減できます。

差分プライベート確率的勾配降下法（DP-SGD）と呼ばれるこのアプローチの基本的な考え方は、ほぼすべてのディープラーニングアルゴリズムの中核にある確率的勾配降下法（SGD）で使用される勾配を変更することです。DP-SGD でトレーニングされたモデルは、入力データに対して証明可能な差分プライバシー保証を提供します。通常の（バニラ）SGD アルゴリズムに対して、次の 2 つの修正が行われます。

1. まず、各勾配の感度を制限（有界に）する必要があります。言い換えると、ミニバッチでサンプリングされた個々のトレーニングデータポイントが、勾配計算およびモデルパラメータに適用される結果の更新にどれだけ影響を与えるかを制限する必要があります。これは、各トレーニングデータポイントで計算された各勾配を*クリッピング*することによって行われます。
2. *ランダムノイズ*をサンプリングしてクリッピングされた勾配に追加します。これにより、トレーニングデータセットにその特定のデータポイントが含まれている場合と含まれていない場合で SGD が適用する更新を比較しても、特定のデータポイントがトレーニングデータセットに含まれていたかどうかを統計的に知ることが不可能になります。

このチュートリアルでは、[tf.keras](https://www.tensorflow.org/guide/keras) を使用して、TensorFlow Privacy ライブラリが提供する DP-SGD オプティマイザで手書き数字を認識する畳み込みニューラルネットワーク（CNN）をトレーニングします。TensorFlow Privacy は、既存の TensorFlow オプティマイザをラップして DP-SGD を実装した派生版を作成するコードを提供します。

## セットアップ
このラボを実行するには特別なカーネルが必要です。次のセルを実行してください。

In [ ]:
!echo "Kernel installation started."
!cd ../../../../.. && make tf_privacy_kernel > /dev/null 2>&1
!echo "Kernel installation completed."

完了したら、ノートブックを先に進める前に、右上の **`TF Privacy Kernel`** を選択してください。<br>
インストール後、カーネルが表示されるまでに約 1 分かかる場合があります。

必要なライブラリをインポートしましょう:

In [1]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

In [3]:
import numpy as np
import tensorflow as tf

tf.get_logger().setLevel("ERROR")

2025-09-16 22:00:22.222336: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-16 22:00:22.222408: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-16 22:00:22.222447: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow Privacy をインポートします。

In [4]:
import tensorflow_privacy
from tensorflow_privacy.privacy.analysis import compute_dp_sgd_privacy

## データセットの読み込みと前処理

[MNIST](http://yann.lecun.com/exdb/mnist/) データセットを読み込み、トレーニング用にデータを準備します。

In [5]:
train, test = tf.keras.datasets.mnist.load_data()
train_data, train_labels = train
test_data, test_labels = test

train_data = np.array(train_data, dtype=np.float32) / 255
test_data = np.array(test_data, dtype=np.float32) / 255

train_data = train_data.reshape(train_data.shape[0], 28, 28, 1)
test_data = test_data.reshape(test_data.shape[0], 28, 28, 1)

train_labels = np.array(train_labels, dtype=np.int32)
test_labels = np.array(test_labels, dtype=np.int32)

train_labels = tf.keras.utils.to_categorical(train_labels, num_classes=10)
test_labels = tf.keras.utils.to_categorical(test_labels, num_classes=10)

assert train_data.min() == 0.0
assert train_data.max() == 1.0
assert test_data.min() == 0.0
assert test_data.max() == 1.0

11490434/11490434 [==============================] - 0s 0us/step


## ハイパーパラメータの定義
学習モデルのハイパーパラメータ値を設定します。

DP-SGD には、調整が必要な 3 つの一般的なハイパーパラメータと 3 つのプライバシー固有のハイパーパラメータがあります。

**一般的なハイパーパラメータ**

1. `epochs`（int） - アルゴリズムによるトレーニングデータの全体的な 1 パスを指します。エポック数が大きいほど、モデルが同じデータポイントで複数回トレーニングされるため、プライバシーリスクが増加します。
2. `batch_size`（int） - バッチサイズは、DP-SGD トレーニングのさまざまな側面に影響を与えます。例えば、バッチサイズを大きくすると、同じプライバシー保証の下でトレーニング中に追加されるノイズの量を減らすことができ、トレーニングの分散が減少します。
3. `learning_rate`（float） - このハイパーパラメータは通常の SGD にも存在します。学習率が高いほど、各更新の影響が大きくなります。更新にノイズが多い場合（クリッピングしきい値に対して加法ノイズが大きい場合など）、学習率を低くするとトレーニング手順の収束に役立つ場合があります。

**プライバシー固有のハイパーパラメータ**
1. `l2_norm_clip`（float） - モデルパラメータの更新に適用される各勾配の最大ユークリッド（L2）ノルム。このハイパーパラメータは、個々のトレーニングポイントに対するオプティマイザの感度を有界にするために使用されます。
2. `noise_multiplier`（float） - クリッピングノルムに対する標準偏差の比率（トレーニング中にサンプリングされ勾配に追加されるノイズの量）。一般に、ノイズが多いほどプライバシーが向上します（多くの場合、有用性の低下を犠牲にしますが、必ずしもそうとは限りません）。
3. `microbatches`（int） - データの各バッチは、マイクロバッチと呼ばれる小さな単位に分割されます。デフォルトでは、各マイクロバッチには単一のトレーニングサンプルが含まれている必要があります。これにより、ミニバッチ全体で平均化された後ではなく、サンプル単位で勾配をクリッピングできます。これにより、勾配に見られるシグナルに対するクリッピングの（悪）影響が軽減され、通常は有用性が最大化されます。ただし、マイクロバッチのサイズを増やして複数のトレーニングサンプルを含めることで、計算オーバーヘッドを削減できます。その場合、これら複数のトレーニングサンプルの平均勾配がクリッピングされます。バッチで消費されるサンプルの総数（つまり勾配降下の 1 ステップ）は同じままです。マイクロバッチ数は、バッチサイズを割り切れる数にする必要があります。


ある程度精度の高いモデル（テスト精度 95%）を取得するには、以下のハイパーパラメータ値を使用します。

In [6]:
epochs = 1
batch_size = 32
learning_rate = 0.25

l2_norm_clip = 1.0
noise_multiplier = 0.5
num_microbatches = 32  # バッチサイズと同じ（つまりマイクロバッチなし）

if batch_size % num_microbatches != 0:
    raise ValueError(
        "Batch noise_multipliere should be an integer multiple of the number of microbatches"
    )

## モデルの構築

学習モデルとして畳み込みニューラルネットワークを定義します。

In [7]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Conv2D(
            16,
            8,
            strides=2,
            padding="same",
            activation="relu",
            input_shape=(28, 28, 1),
        ),
        tf.keras.layers.MaxPool2D(2, 1),
        tf.keras.layers.Conv2D(
            32, 4, strides=2, padding="valid", activation="relu"
        ),
        tf.keras.layers.MaxPool2D(2, 1),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(10, activation="softmax"),
    ]
)

学習モデルのオプティマイザと損失関数を定義します。各トレーニングポイントに対する勾配操作をサポートするために、ミニバッチ全体の平均としてではなく、サンプルごとの損失のベクトルとして損失を計算します。

In [8]:
optimizer = tensorflow_privacy.DPKerasSGDOptimizer(
    l2_norm_clip=l2_norm_clip,
    noise_multiplier=noise_multiplier,
    num_microbatches=num_microbatches,
    learning_rate=learning_rate,
)

loss = tf.keras.losses.CategoricalCrossentropy(
    reduction=tf.losses.Reduction.NONE
)

## モデルのトレーニング

In [9]:
model.compile(optimizer=optimizer, loss=loss, metrics=["accuracy"])

model.fit(
    train_data,
    train_labels,
    epochs=epochs,
    validation_data=(test_data, test_labels),
    batch_size=batch_size,
)

1875/1875 [==============================] - 145s 76ms/step - loss: 0.8429 - accuracy: 0.8256 - val_loss: 0.9353 - val_accuracy: 0.8737


## 差分プライバシー保証の測定

プライバシー分析を実行して、トレーニングアルゴリズムによって達成された DP 保証を測定します。達成された DP のレベルを知ることで、2 つのトレーニング実行を客観的に比較し、どちらがよりプライバシーを保護しているかを判断できます。高レベルでは、プライバシー分析は、潜在的な攻撃者がトレーニング手順の結果（例: モデルの更新やパラメータ）を観察することによって、個々のトレーニングポイントの特性に関する推測をどれだけ改善できるかを測定します。

この保証は、**プライバシーバジェット（privacy budget）**と呼ばれることもあります。プライバシーバジェットが低いほど、推測を改善する攻撃者の能力がより厳密に制限されます。これにより、より強力なプライバシー保証が確保されます。直感的には、単一のトレーニングポイントが学習の結果に影響を与えることが難しくなるためです。例えば、トレーニングポイントに含まれる情報を ML アルゴリズムが記憶することができず、データセットにそのトレーニングポイントを提供した個人のプライバシーが保護されます。

このチュートリアルでは、[こちらの論文](https://arxiv.org/abs/1702.07476) に基づく純粋な DP の緩和であり、DP-SGD に特に適している Rényi 差分プライバシー（Rényi Differential Privacy: RDP）の枠組みでプライバシー分析を実行します。

ML アルゴリズムの DP 保証を表すために、2 つの指標が使用されます。

1.   デルタ（$\delta$） - プライバシー保証が成り立たない確率を有界にします。経験則としては、トレーニングデータセットのサイズの逆数未満に設定することです。このチュートリアルでは、MNIST データセットに 60,000 個のトレーニングポイントがあるため、$10^{-5}$ に設定されています。
2.   イプシロン（$\epsilon$） - これはプライバシーバジェットです。単一のトレーニングポイントを含める（または除外する）ことによって特定のモデル出力の確率がどれだけ変化し得るかを有界にすることで、プライバシー保証の強度（または入力データに関する情報を明らかにする最大許容範囲）を測定します。$\epsilon$ の値が小さいほど、より優れたプライバシー保証を意味します。ただし、$\epsilon$ の値は上限に過ぎず、値が大きくても実際には十分なプライバシーが保たれている場合があります。

$(\epsilon, \delta)$-差分プライバシーの数学的定義の詳細については、オリジナルの [DP-SGD 論文](https://arxiv.org/pdf/1607.00133.pdf) を参照してください。

TensorFlow Privacy は、固定された $\delta$ の値とトレーニングプロセスからの以下のハイパーパラメータが与えられた場合に $\epsilon$ の値を計算するツール `compute_dp_sgd_privacy` を提供します。

1.   トレーニングデータ内のデータポイントの総数、`n`。
2. `batch_size`。
3.   `noise_multiplier`。
4. トレーニングの `epochs` 数。

In [10]:
dpsgd_statement = compute_dp_sgd_privacy.compute_dp_sgd_privacy_statement(
    number_of_examples=train_data.shape[0],
    batch_size=batch_size,
    noise_multiplier=noise_multiplier,
    used_microbatching=False,
    num_epochs=epochs,
    delta=1e-5,
)

print(dpsgd_statement)

DP-SGD performed over 60000 examples with 32 examples per iteration, noise
multiplier 0.5 for 1 epochs without microbatching, and no bound on number of
examples per user.

This privacy guarantee protects the release of all model checkpoints in addition
to the final model.

Example-level DP with add-or-remove-one adjacency at delta = 1e-05 computed with
RDP accounting:
    Epsilon with each example occurring once per epoch:        10.726
    Epsilon assuming Poisson sampling (*):                      3.800

No user-level privacy guarantee is possible without a bound on the number of
examples per user.

(*) Poisson sampling is not usually done in training pipelines, but assuming
that the data was randomly shuffled, it is believed the actual epsilon should be
closer to this value than the conservative assumption of an arbitrary data
order.



このツールは、$\delta=10^{-5}$ を含め、上記で選択したハイパーパラメータに対する $\epsilon$ の値を報告します。

Copyright 2025 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License